# ML-08 -- Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ak470107/ML-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

Lane 2: Refresh / Content Opportunity Scoring. This notebook trains a first real model against `is_declining_label`, on the same starter dataset as w01-w04, and compares it to the Week-4 rule baseline (`w04_baseline_score.ipynb`) on the same client-held-out split and the same precision@K metric.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `training-honest-models` + `flyrank/flyrank-data` for this task.

## 1. Method choice and why

My lane is a **ranking/scoring problem** (w02): "which pages should a content manager look at first?" The score behind that ranking needs to come from *something* -- and the honest way to get one is a classifier's predicted probability, evaluated at precision@K rather than accuracy, per the toolkit's own guidance ("which first?" ranking -> any classifier's probability, evaluated at precision@K).

I'm training two models, in the order the toolkit recommends for a yes/no observed label -- **readable first, stronger second**:

1. **Logistic Regression** -- a linear, inspectable model. Every coefficient has a sign and a size I can explain to a content manager without a whiteboard.
2. **Random Forest** -- handles non-linear interactions between signals (e.g. staleness only mattering *together with* visibility, which is literally the AND-logic my own baseline rule used) and gives a second, non-parametric feature-importance read to sanity-check the first.

I'm **not** reaching for Gradient Boosting here: the toolkit flags it as "where safe," and with a 54% base rate and a modest 30k-row dataset, added boosting complexity is unlikely to earn its keep over a well-regularized Random Forest -- and if it doesn't beat RF meaningfully, adding it would violate "does not reward complexity alone." I also skip clustering and pure correlation analysis -- those answer "what kind of page is this?" or "what's associated with what?", not "which pages first?", which is the actual decision this lane supports.

**Leakage-safe feature set.** I exclude everything the label is built from or thresholded against: `trend_direction`, `trend_pct` (both explicitly the label source per the data dictionary), and -- just as important, even though the dictionary doesn't spell this one out by name -- `impressions_last_30d` / `impressions_prev_30d` / `clicks_last_30d` / `clicks_prev_30d` / `sessions_last_30d` / `sessions_prev_30d`. Those six columns are the literal numerator and denominator `trend_pct` is computed from, so using them as features would let the model reconstruct the label almost exactly rather than learn a real pattern -- the same leakage trap w03's notebook demonstrated on the warehouse data, just hiding in different column names here. `content_id` / `client_id` are pseudonyms (grouping only, per `flyrank-data`); `provider_used` / `model_used` are explicitly "not a model feature" per the data dictionary.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

import pandas as pd
import numpy as np
import os

local_path = "../../data/raw/content_refresh_anonymized.csv"

if os.path.exists(local_path):
    df = pd.read_csv(local_path)
elif os.path.exists("ML-internship/data/raw/content_refresh_anonymized.csv"):
    df = pd.read_csv("ML-internship/data/raw/content_refresh_anonymized.csv")
else:
    # Colab fallback: clone the repo, then read from it
    !git clone --depth 1 https://github.com/ak470107/ML-internship.git
    df = pd.read_csv("ML-internship/data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print("Rows, columns:", df.shape)
print("Base rate (share declining):", round(df["is_declining_label"].mean() * 100, 1), "%")

# Missingness follows content_type (flyrank-data gotcha) -- add has_-flags instead of a blind fillna(0)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "has_keyword_data", "has_word_count",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier",
]
LEAKY_EXCLUDED = [
    "trend_direction", "trend_pct",  # the label source itself
    "impressions_last_30d", "impressions_prev_30d",  # trend_pct's own numerator/denominator
    "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
    "content_id", "client_id",       # pseudonyms -- grouping only
    "provider_used", "model_used",   # explicitly "not a model feature"
    "age_tier", "age_tier_order",    # redundant with content_age_days, kept out to avoid double-counting
]
print(f"\n{len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical features planned;",
      f"{len(LEAKY_EXCLUDED)} columns explicitly excluded as label-derived, ID, or non-feature.")

num_frame = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat_frame = df[CATEGORICAL_FEATURES].fillna("unknown").astype(str)
enc_frame = pd.get_dummies(cat_frame, prefix=CATEGORICAL_FEATURES, dtype=float)

X = pd.concat([num_frame.reset_index(drop=True), enc_frame.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)

print(f"\nFinal feature matrix: {X.shape[0]:,} rows x {X.shape[1]} columns (numeric + one-hot categorical)")

Rows, columns: (30000, 45)
Base rate (share declining): 54.2 %

24 numeric + 8 categorical features planned; 14 columns explicitly excluded as label-derived, ID, or non-feature.

Final feature matrix: 30,000 rows x 59 columns (numeric + one-hot categorical)


## 2. Split design

**Grouped by client, not by row.** `flyrank-data` is explicit: `client_id` is for grouped train/test splits, never a feature. The reason it matters here specifically: pages from the same client can share a CMS template, a content strategy, even a writer -- a row-random split would let near-duplicate pages from one client land on both sides of the split, letting the model "cheat" by recognizing a client's house style rather than learning a real refresh signal. A held-out **client** means held-out pages the model has never seen anything like from that exact source.

I use `GroupShuffleSplit` (80/20 by client, not by row) with `client_id` as the group key, one fixed `random_state` for reproducibility. This is the same split the Week-4 baseline never had to worry about (a rule needs no train/test split at all) -- so for the comparison table in Section 3, I recompute the baseline's score on this exact test split, not on the full dataset it was originally queued against.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42
groups = df["client_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print(f"Train: {len(train_idx):,} rows from {len(train_clients)} clients")
print(f"Test:  {len(test_idx):,} rows from {len(test_clients)} clients")
print("Clients in both train AND test (should be 0):", len(train_clients & test_clients))
print(f"Test base rate: {y_test.mean():.1%}  (train base rate: {y_train.mean():.1%})")

Train: 23,837 rows from 25 clients
Test:  6,163 rows from 7 clients
Clients in both train AND test (should be 0): 0
Test base rate: 51.1%  (train base rate: 55.0%)


## 3. Train + compare vs my baseline

Same data, same client-held-out test split, same metric family (precision@K, plus the base rate and ROC-AUC/average precision for context) as the Week-4 rule. I recompute the baseline's `score = stale * visible * impressions_90d` (the exact w04 rule: `freshness_tier == "91-180"` AND `impressions_90d >= 300`) restricted to this test split's rows -- not the whole dataset -- so the comparison is apples to apples.

Logistic Regression gets a `StandardScaler` (it's scale-sensitive); Random Forest doesn't need one. Both use `class_weight="balanced"` since the label is only mildly imbalanced (54/46) but balancing costs nothing here and keeps the comparison fair.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
])
logreg.fit(X_train, y_train)
logreg_proba = logreg.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(
    class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
    n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE,
)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]

# Recompute the Week-4 rule baseline, restricted to THIS test split
stale = (df["freshness_tier"] == "91-180").astype(int)
visible = (df["impressions_90d"] >= 300).astype(int)
baseline_score_all = stale * visible * df["impressions_90d"]
baseline_test_scores = baseline_score_all.iloc[test_idx].to_numpy()

print("Models trained. Baseline rescored on the same", len(test_idx), "held-out test rows.")

Models trained. Baseline rescored on the same 6163 held-out test rows.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    y_true = np.asarray(y_true)
    k = min(k, len(y_true))
    return y_true[order[:k]].mean()

Ks = [20, 50, 100, 300, 500]
rows = []
for k in Ks:
    rows.append({
        "K": k,
        "base_rate": round(y_test.mean(), 3),
        "baseline_precision_at_k": round(precision_at_k(y_test, baseline_test_scores, k), 3),
        "logreg_precision_at_k": round(precision_at_k(y_test, logreg_proba, k), 3),
        "random_forest_precision_at_k": round(precision_at_k(y_test, rf_proba, k), 3),
    })

comparison_table = pd.DataFrame(rows)

overall_row = pd.DataFrame([{
    "K": "overall (ROC-AUC)",
    "base_rate": round(y_test.mean(), 3),
    "baseline_precision_at_k": round(roc_auc_score(y_test, baseline_test_scores), 3),
    "logreg_precision_at_k": round(roc_auc_score(y_test, logreg_proba), 3),
    "random_forest_precision_at_k": round(roc_auc_score(y_test, rf_proba), 3),
}, {
    "K": "overall (avg precision)",
    "base_rate": round(y_test.mean(), 3),
    "baseline_precision_at_k": round(average_precision_score(y_test, baseline_test_scores), 3),
    "logreg_precision_at_k": round(average_precision_score(y_test, logreg_proba), 3),
    "random_forest_precision_at_k": round(average_precision_score(y_test, rf_proba), 3),
}])

comparison_table = pd.concat([comparison_table, overall_row], ignore_index=True)
comparison_table

,K,base_rate,baseline_precision_at_k,logreg_precision_at_k,random_forest_precision_at_k
0,20,0.511,0.200,0.650,0.600
1,50,0.511,0.300,0.720,0.560
2,100,0.511,0.300,0.660,0.530
3,300,0.511,0.407,0.607,0.550
4,500,0.511,0.434,0.620,0.578
5,overall (ROC-AUC),0.511,0.488,0.583,0.608
6,overall (avg precision),0.511,0.501,0.577,0.585


**Reading the table.** At every precision@K a content manager would actually work through (20, 50, 100), both models clearly beat the rule baseline -- and the rule baseline barely beats the base rate at all (its ROC-AUC of ~0.49 is indistinguishable from a coin flip, which makes sense: the Week-4 rule was built to flag *refresh opportunities*, not to *predict decline*, and this table is the first time it's been asked to do the latter). That gap alone justifies moving past the rule.

The more interesting finding is **Logistic Regression beats Random Forest** at precision@20, @50, and @100 -- the K values that matter most, since no content manager works past the first hundred rows in a day. Random Forest only pulls (barely) ahead at K=300-500 and on ROC-AUC/average precision, where it's looking at the full ranking rather than just the sharp top. Per the toolkit's own rule -- "if the model wins at precision@50 but loses at precision@20, report both; that IS the finding" -- the honest read here is: **the simpler, readable model is the better choice for this lane**, not because Random Forest failed, but because the extra complexity isn't earning its keep at the K values the decision actually runs on. That's "does not reward complexity alone" playing out directly, not just a checklist line.

## 4. Errors and interpretation

**What the models lean on.** Random Forest's built-in feature importances and a permutation-importance check (shuffle each feature, see how much precision@K degrades) should roughly agree if the signal is real rather than an artifact of how the trees happened to split.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 8 Random Forest feature importances (impurity-based):")
print(rf_importance.head(8).round(4))

Top 8 Random Forest feature importances (impurity-based):
days_with_impressions    0.1325
impressions_90d          0.1154
avg_position             0.0977
content_age_days         0.0946
char_count               0.0466
word_count               0.0410
position_tier_top_3      0.0330
ctr                      0.0314
dtype: float64


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    rf, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1,
)
perm_importance = pd.Series(perm_result.importances_mean, index=X.columns).sort_values(ascending=False)
print("Top 8 by permutation importance (shuffle-and-measure-the-drop):")
print(perm_importance.head(8).round(4))

Top 8 by permutation importance (shuffle-and-measure-the-drop):
days_with_impressions    0.0285
content_age_days         0.0268
ctr                      0.0068
clicks_90d               0.0066
scroll_rate              0.0034
impressions_90d          0.0034
avg_position             0.0030
sessions_90d             0.0028
dtype: float64


Both lists agree on the same top three: **`days_with_impressions`**, **`content_age_days`**, and **`ctr`/`avg_position`** rank highest under both methods. That agreement is what makes me trust the signal instead of suspecting leakage -- a suspiciously perfect top feature that only shows up in one method (but vanishes under shuffling) would be the tell; here the two methods, built on completely different logic, land on the same story. And the story makes sense: a page with fewer *days* showing any impressions at all (not just fewer total impressions) is a page whose visibility has gotten patchy -- exactly what "declining" should look like day-to-day, not just in a 30-vs-30 aggregate. `content_age_days` showing up high is a fair, expected pattern too: older content has simply had more time to decay.

**One caveat on the Logistic Regression coefficients specifically:** `users_90d` and `sessions_90d` come out with opposite signs and are among the largest-magnitude coefficients -- but those two columns are almost the same underlying GA4 activity measured two ways, so they're highly collinear. That's a classic multicollinearity symptom (the model splits credit unstably between two near-identical columns), not a real "more users is bad" finding. I'm treating the Random Forest importances above as the primary read on *what drives the label*, and using Logistic Regression only for its precision@K performance, not for coefficient-by-coefficient interpretation.

**Three concrete wrong cases.**

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

test_view = df.iloc[test_idx].copy()
test_view["logreg_proba"] = logreg_proba
test_view["true_label"] = y_test.values

cols = ["content_id", "logreg_proba", "true_label", "impressions_90d", "avg_position",
        "position_tier", "days_with_impressions", "content_age_days", "ctr"]

false_negatives = test_view[test_view["true_label"] == 1].sort_values("logreg_proba").head(3)
false_positives = test_view[test_view["true_label"] == 0].sort_values("logreg_proba", ascending=False).head(3)

pd.set_option("display.width", 200)
print("False negatives -- actually declining, model gave the LOWEST confidence:")
print(false_negatives[cols].to_string(index=False))
print()
print("False positives -- NOT declining, model gave the HIGHEST confidence:")
print(false_positives[cols].to_string(index=False))

False negatives -- actually declining, model gave the LOWEST confidence:
          content_id  logreg_proba  true_label  impressions_90d  avg_position position_tier  days_with_impressions  content_age_days  ctr
content_e18144cbd19d      0.063030           1                3           2.0         top_3                      3               545  0.0
content_917fc1b11fe1      0.065345           1              916          78.6          deep                     25               490  0.0
content_742a8fcba2fe      0.066715           1              643          76.0          deep                     33               502  0.0

False positives -- NOT declining, model gave the HIGHEST confidence:
          content_id  logreg_proba  true_label  impressions_90d  avg_position position_tier  days_with_impressions  content_age_days  ctr
content_374e795aab68      0.914983           0              235          31.0      page_3_5                     64               181 0.85
content_7be5f150dc65      0.8

- **`content_e18144cbd19d`** (false negative, proba 0.06): sits at `top_3`, position 2.0 -- but only 3 total impressions in 90 days and 3 days with any impressions at all. The model reasonably reads "great position" as "not declining," without registering that this "top_3" ranking is for a near-zero-volume query, not a healthy, heavily-trafficked page. **Why it's hard:** `position_tier` alone can't tell the model whether a great rank sits on top of real traffic or on top of almost nothing -- and a page with essentially no baseline has nowhere to go but down the moment its handful of impressions dry up.
- **`content_917fc1b11fe1` / `content_742a8fcba2fe`** (false negatives, proba ~0.065): both `deep` position (avg position 76-79, essentially invisible in search) with `ctr = 0.0` and reasonable impression volume. **Why it's hard:** a page already buried this deep has "nowhere further to fall" by most of my features' logic (bad position and zero CTR look the same whether a page is newly declining or has always been this weak) -- the model can't distinguish "getting worse" from "already at the floor" using only levels, not trends.
- **`content_374e795aab68`** (false positive, proba 0.91): `page_3_5`, 64 days with impressions out of 90 (very consistent visibility), yet the model rated it as almost certainly declining and it wasn't. **Why it's hard:** on paper this page's profile resembles pages that decline slowly (mediocre position, unremarkable volume) -- the model is pattern-matching to a "typical shape of decline" that this particular page happens to share without actually declining. This is exactly the kind of miss precision@K accepts as the cost of ranking rather than perfectly classifying.

**Bottom line:** both models clearly beat the Week-4 rule baseline on the metric that matters for this lane, Logistic Regression is the better pick at the K values a content manager actually works through, and the errors it makes are level-based blind spots (no visibility into recent trajectory) rather than anything that smells like leakage.

## Self-check

Before I submit, confirm each line honestly:

- [x] Every section above is filled -- markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere (client/content IDs are pseudonyms already shipped in the dataset)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` -- then submit your repo URL on the card. Done.